In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# Simulate imbalanced dataset
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'age':       np.random.randint(18, 65, n),
    'salary':    np.random.randint(20000, 120000, n),
    'education': np.random.choice(['high_school', 'bachelors', 'masters'], n),
    'churned':   np.random.choice([0, 1], n, p=[0.95, 0.05])  # 95/5 split
})

print("Class distribution:")
print(df['churned'].value_counts())
print(f"\nIf model predicts 0 always: {df['churned'].value_counts()[0]/n*100:.1f}% accuracy")

Class distribution:
churned
0    949
1     51
Name: count, dtype: int64

If model predicts 0 always: 94.9% accuracy


# Churned - someone who stopped using a subscription
94.9% accuracy means nothing here because a model that predicts "not churned" for every single customer still scores 94.9%. It never catches a single customer who actually churned. In a real business that means 51 customers leave and nobody notices until it's too late

In [3]:
# Your turn — fill in the blanks
# Follow the correct order we drilled

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# Step 1 — Encode
le = LabelEncoder()
df['education_encoded'] = le.fit_transform(df['education'])

# Step 2 — Features and target
X = df[['age', 'salary', 'education_encoded']]
y = df['churned']

# Step 3 — Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 4 — SMOTE on training only
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {y_train_balanced.value_counts().to_dict()}")

# Step 5 — Train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_balanced, y_train_balanced)

# Step 6 — Evaluate
print(classification_report(y_test, model.predict(X_test)))

Before SMOTE: {0: 759, 1: 41}
After SMOTE:  {0: 759, 1: 759}
              precision    recall  f1-score   support

           0       0.95      0.81      0.87       190
           1       0.05      0.20      0.08        10

    accuracy                           0.78       200
   macro avg       0.50      0.50      0.48       200
weighted avg       0.91      0.78      0.83       200

